[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/purple/notebooks/purple_natural_product_screen.ipynb)

# Screening African natural products with the HIV-1 model

**Purple group · HIV**

In `purple_baseline_models` we trained a model to tell potent anti-HIV compounds from
weak ones. Now we point it at ANPDB, a database of molecules isolated from African
plants, and ask which of them are worth testing. Most of this notebook is about
deciding whether the answer can be believed.

## What you will do

- Load ANPDB and DrugBank, and clean them the way the training data was cleaned.
- Measure how far these molecules sit from anything the model has seen.
- Find the molecules the model was already trained on, which must not be reported as findings.
- Score both libraries and look at what comes out on top.
- Compare what the classifier says with what the regression model says.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "purple"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. The two libraries we are going to screen

**ANPDB** is the African Natural Products Database: compounds isolated from plants
used across the continent. These are the molecules the group cares about, and almost
none of them have ever been tested against HIV.

**DrugBank** is a collection of approved and experimental medicines. We are not hoping
to discover anything in it. It is there as a yardstick: without something to compare
against, a score of 0.6 on a natural product means nothing at all.

Both files live in the project's `data` folder. Let us see how big they are.

In [ ]:
import numpy as np
import pandas as pd
import stylia
from scripts import curation, modelling, screening

RANDOM_SEED = 42
THRESHOLD = 6.0  # pActivity 6 is the 1 uM cutoff the models were trained with

anpdb_raw = pd.read_csv("data/anpdb_smiles.csv")
drugbank_raw = pd.read_csv("data/drugbank_smiles.csv")
print(f"ANPDB:    {len(anpdb_raw):,} rows")
print(f"DrugBank: {len(drugbank_raw):,} rows")

Real data is never quite as advertised. Two columns of this ANPDB export are
completely empty, and several hundred rows have no molecule in them at all.

In [ ]:
print("empty columns:", [c for c in anpdb_raw.columns if anpdb_raw[c].isna().all()])
print("rows with no SMILES:", int(anpdb_raw["smiles"].isna().sum()))

anpdb_raw = anpdb_raw.drop(columns=["Region", "InChI_Key"])
anpdb_raw.head(3)

## 2. Clean the molecules the way the training data was cleaned

A model does not really take a molecule as input. It takes a fingerprint, computed
from a SMILES string that was written in one particular way. If we prepare these
libraries differently from how the training set was prepared, we are asking the model
questions in a language it was never taught.

So we run both libraries through the same steps as `purple_data_curation`: remove
anything RDKit cannot read, strip salts down to the parent molecule, rewrite every
structure in one agreed form, compute an InChIKey, and drop anything above 1000 g/mol,
which is the limit the models were trained behind. `screening.prepare_library` does
all of that.

ANPDB first. This takes about twenty seconds.

In [ ]:
anpdb = screening.prepare_library(
    anpdb_raw, smiles_column="smiles", id_column="molecule_id",
    name_column="mol_name", label="ANPDB",
)
print(f"{len(anpdb_raw):,} rows in the file -> {len(anpdb):,} unique molecules")
anpdb.head(3)

Now DrugBank. Watch the count drop further here: DrugBank contains
peptides and other biological drugs that are far heavier than 1000 g/mol, and the
weight limit removes them. That is the right thing to do, because the model never
saw molecules like that.

In [ ]:
drugbank = screening.prepare_library(
    drugbank_raw, smiles_column="Smiles", id_column="DrugBankId",
    name_column=None, label="DrugBank",
)
print(f"{len(drugbank_raw):,} rows in the file -> {len(drugbank):,} unique molecules")

library = pd.concat([anpdb, drugbank], ignore_index=True)
print(f"{len(library):,} molecules to screen in total")

Finally we turn every molecule into a fingerprint, using exactly the
settings the models were trained with: Morgan fingerprints of radius 2 and 2048 bits.
Changing either number here would silently break everything that follows.

In [ ]:
X_library = modelling.morgan_fingerprints(library["smiles"], radius=2, n_bits=2048)
print("fingerprints:", X_library.shape)

## 3. How far are these molecules from anything the model has seen?

A model learns from examples. Asked about something that looks nothing like its
examples, it still returns a number, and that number is guesswork dressed up as a
prediction. So before we score anything, we measure the distance.

For each library molecule we find its **nearest neighbour** in the training set and
record how similar the two are. The measure is the Tanimoto coefficient: the fraction
of fingerprint bits two molecules share. It runs from 0 for nothing in common to 1
for identical fingerprints. As a rough guide, two unrelated molecules score around
0.1, and anything above 0.6 usually looks clearly related to a chemist.

First we rebuild the training set exactly as the modelling notebook had it.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

curated = pd.read_csv("data/hiv1_curated.csv")
curated["mw"] = [Descriptors.MolWt(Chem.MolFromSmiles(s)) for s in curated["smiles"]]
curated = curated[curated["mw"] <= 1000].reset_index(drop=True)
X_train = modelling.morgan_fingerprints(curated["smiles"], radius=2, n_bits=2048)
print(f"{len(curated):,} training molecules, {curated['activity'].mean():.1%} of them active")

Now the comparison itself: every library molecule against every training
molecule. That is about 450 million pairs, and it finishes in a couple of seconds
because the whole thing is one matrix multiplication.

In [ ]:
similarity, nearest = screening.max_similarity(X_library, X_train)
library["max_similarity"] = similarity

for name, group in library.groupby("library"):
    share = (group["max_similarity"] < 0.4).mean()
    print(f"{name:9s} median similarity to training {group['max_similarity'].median():.3f}"
          f"   {share:.0%} below 0.4")

It is worth seeing this as a picture. The grey curve is the training set
compared against itself, which is what "close to home" looks like. Both libraries sit
well to the left of it.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

# Split the training set in two and compare the halves, to see what "close" looks like.
train_similarity, _ = screening.max_similarity(X_train[:4000], X_train[4000:])

fig, axs = stylia.create_figure(1, 1, width=0.5)
ax = axs.next()
ax.hist(train_similarity, bins=50, range=(0, 1), density=True, histtype="step",
        linewidth=2, color=nc.gray, label="training set")
for name, color in [("DrugBank", nc.mint), ("ANPDB", nc.purple)]:
    ax.hist(library.loc[library["library"] == name, "max_similarity"], bins=50,
            range=(0, 1), density=True, histtype="step", linewidth=2,
            color=color, label=name)
ax.legend()
stylia.label(ax, xlabel="Similarity to nearest training molecule", ylabel="Density",
             title="How far the libraries sit from the training data")

> **Note:** A low similarity is not a reason to throw a molecule away. It is
a reason to treat its score as a weak hint rather than a prediction. Keep an eye on
this column for the rest of the notebook.

### 3.1 The same picture as a map

Numbers in a table are hard to feel. The Ersilia model
[eos1klk](https://github.com/ersilia-os/eos1klk) places any molecule on a map built
from 1.3 million reference compounds, so we can see all three sets at once. This is
the same map `purple_chemical_space` used, so the coordinates are comparable.

In [ ]:
space = pd.concat([
    pd.read_csv("data/eos1klk_anpdb.csv").assign(library="ANPDB"),
    pd.read_csv("data/eos1klk_drugbank.csv").assign(library="DrugBank"),
], ignore_index=True)
training_space = pd.read_csv("data/eos1klk_hiv1_curated.csv")
print(f"{len(space):,} library molecules placed on the map")
print(f"{len(training_space):,} training molecules already on it")

Each dot is one molecule. The training set is in grey underneath, so you
can see where the libraries do and do not overlap it.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.scatter(training_space["umap_x"], training_space["umap_y"], s=3, alpha=0.3,
           color=nc.gray, linewidths=0, label="HIV-1 training set")
for name, color in [("DrugBank", nc.mint), ("ANPDB", nc.purple)]:
    subset = space[space["library"] == name]
    ax.scatter(subset["umap_x"], subset["umap_y"], s=2, alpha=0.3, color=color,
               linewidths=0, label=name)
ax.legend(markerscale=6)
stylia.label(ax, xlabel="UMAP 1", ylabel="UMAP 2",
             title="Where the libraries sit in chemical space")

## 4. Which molecules has the model already seen?

Here is a trap that catches almost everybody. Some molecules in ANPDB were studied
against HIV years ago, so they are already in the training data. The model knows their
answers by heart. If we leave them in, they come straight to the top of our list and we
congratulate ourselves on rediscovering them.

We already have what we need to find them. A molecule that is identical to one in the
training set has an identical fingerprint, so its similarity is exactly 1.00. That one
number is the whole rule.

In [ ]:
library["already_seen"] = library["max_similarity"] >= 0.999
for name, group in library.groupby("library"):
    print(f"{name:9s} {int(group['already_seen'].sum()):>4} molecules the model has already seen")

These are not candidates. They are answers the model was given during
training. Here are a few of the ANPDB ones.

In [ ]:
seen = library[library["already_seen"] & (library["library"] == "ANPDB")]
seen[["mol_name", "mw", "max_similarity"]].head(8)

> **Note:** A similarity of exactly 1.00 does not always mean the two
> structures are drawn identically. A fingerprint records which small fragments a
> molecule contains, not how many times a plain carbon chain repeats, so two compounds
> differing only in the length of a fatty acid chain share a fingerprint. To the model
> they are the same molecule, and that is what matters here.

Everything else is fair game.

In [ ]:
keep = ~library["already_seen"]
print(f"{int(library['already_seen'].sum()):,} molecules removed as already seen")
print(f"{int(keep.sum()):,} molecules left to score")

## 5. Score both libraries

We re-train the classifier here rather than loading a saved file. It takes a few
seconds, and it guarantees the model matches the data in this repository.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED
).fit(X_train, curated["activity"])
print("trained on", X_train.shape[0], "molecules")

Before reading any score, it is worth knowing what the model was actually
asked to learn. The training set was split into "active" and "inactive" at 1 uM. Look
at where that line falls on the potency of the molecules.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5)
ax = axs.next()
ax.hist(curated["pactivity"], bins=60, range=(3, 11.5), color=nc.purple)
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--", linewidth=2)
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"The cutoff sits at the median (median {curated['pactivity'].median():.2f})")

The line falls almost exactly in the middle of the distribution, which is
why about half the training set is "active". That is not because half of all molecules
inhibit HIV. It is because ChEMBL only contains compounds somebody thought worth
testing and publishing, and their potencies happen to straddle 1 uM.

> **Note:** So the classifier is really answering "does this look like the *potent*
> end of published anti-HIV chemistry?", not "is this an HIV drug?". A score of 0.9
> is a strong resemblance, not a 90% chance of working.

With that in mind, we can score everything. We score the whole library,
including the molecules set aside in section 4, because their scores are needed in a
moment. `screen` holds the ones we are allowed to report as findings.

In [ ]:
library["p_active"] = classifier.predict_proba(X_library)[:, 1]
screen = library[keep].reset_index(drop=True)
X_screen = X_library[keep.to_numpy()]

for name, group in screen.groupby("library"):
    print(f"{name:9s} median {group['p_active'].median():.3f}   "
          f"top 1% above {group['p_active'].quantile(0.99):.3f}   "
          f"{int((group['p_active'] > 0.5).sum()):,} above 0.5")

Now the comparison the DrugBank set was there for. The dashed line marks
the proportion of the training set that was active, so it is the score you would get
by guessing. Both libraries sit well to the left of it, and the natural products sit
furthest left.

In [ ]:
fig, axs = stylia.create_figure(1, 2, width=1.0, height=0.4)

ax = axs.next()
for name, color in [("DrugBank", nc.mint), ("ANPDB", nc.purple)]:
    ax.hist(screen.loc[screen["library"] == name, "p_active"], bins=40, range=(0, 1),
            density=True, histtype="step", linewidth=2, color=color, label=name)
ax.axvline(curated["activity"].mean(), color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="P(active)", ylabel="Density", title="Score distributions")

ax = axs.next()
for name, color in [("DrugBank", nc.mint), ("ANPDB", nc.purple)]:
    subset = screen[screen["library"] == name]
    ax.scatter(subset["max_similarity"], subset["p_active"], s=3, alpha=0.2,
               color=color, linewidths=0, label=name)
ax.legend(markerscale=4)
stylia.label(ax, xlabel="Similarity to training set", ylabel="P(active)",
             title="Score against distance from the training data")

> **Exercise:** Most natural products score *below* the dashed line, lower
> than a coin flip. Why would a model give unfamiliar molecules low scores rather than
> uncertain ones near 0.5? Think about what a decision tree does when a fragment it
> was looking for is absent.

## 6. A check that looks convincing and is not

Before trusting the ranking, we should test the model on molecules whose answer we
already know. There are 26 approved anti-HIV medicines, and DrugBank contains them.
A good model should put them near the top.

In [ ]:
drugs = pd.read_csv("data/hiv_drugs_approved.csv")
drugbank = library[library["library"] == "DrugBank"]
found = drugs.merge(drugbank[["inchikey", "p_active"]], on="inchikey", how="inner")
found["percentile"] = [(drugbank["p_active"] < p).mean() for p in found["p_active"]]

print(f"{len(found)} of the {len(drugs)} approved drugs are in DrugBank")
print(f"median percentile among all {len(drugbank):,} drugs: {found['percentile'].median():.3f}")
print(f"in the top 1%: {int((found['percentile'] >= 0.99).sum())} of {len(found)}")

That looks like an excellent result. It is worthless. Check where those
molecules came from.

In [ ]:
in_training = drugs["inchikey"].isin(set(curated["inchikey"]))
print(f"approved HIV drugs that are in the training set: {int(in_training.sum())} of {len(drugs)}")

Every single one. The model was trained on these molecules and their
measured activities, so ranking them highly shows only that it can remember. This is
the oldest mistake in machine learning: **a model tested on its training data always
looks good.**

It is also why section 4 mattered. The molecules we removed there would have flattered
the ANPDB results in exactly the same way.

> **Note:** A real check needs molecules the model has never met. One honest version
> is to remove each drug *and everything resembling it* from the training data, retrain,
> and only then ask where the drug ranks. That is worth trying as an exercise.

## 7. What the model picked

It is worth seeing what section 4 saved us from. Here is the top of the ANPDB list if
we had skipped that step.

In [ ]:
naive = (library[library["library"] == "ANPDB"]
         .sort_values("p_active", ascending=False)
         .head(5))
naive[["mol_name", "p_active", "max_similarity", "already_seen"]].round(3)

Five molecules with a near-perfect score, every one of them already in the
training data. That is what a result looks like when a model is quietly grading its own
homework.

Now the real list, with those removed and the similarity column beside each score so we
can see how much of a guess each one is.

In [ ]:
top = (screen[screen["library"] == "ANPDB"]
       .sort_values("p_active", ascending=False)
       .head(20)
       .reset_index(drop=True))
top[["mol_name", "p_active", "max_similarity", "mw"]].round(3)

It is much easier to judge molecules by looking at them.

In [ ]:
from scripts import chemspace

legends = [f"{name[:24]}\np={p:.2f}" for name, p in zip(top["mol_name"], top["p_active"])]
chemspace.draw_molecules(top["smiles"].head(12), legends[:12], per_row=4)

Twenty rows look like twenty findings. They are not. Notice how many of
those structures share the same complicated cage of fused rings, with only the
attached chains differing.

We can measure that instead of eyeballing it: compare how similar the top 20 are to
*each other* with how similar 20 randomly chosen natural products are.

In [ ]:
X_top = modelling.morgan_fingerprints(top["smiles"], radius=2, n_bits=2048)
random_pick = screen[screen["library"] == "ANPDB"].sample(20, random_state=RANDOM_SEED)
X_random = modelling.morgan_fingerprints(random_pick["smiles"], radius=2, n_bits=2048)

top_similarity = np.median(screening.self_similarity(X_top))
random_similarity = np.median(screening.self_similarity(X_random))
print(f"median similarity within the top 20:      {top_similarity:.3f}")
print(f"median similarity within 20 random ANPDB: {random_similarity:.3f}")
print(f"the top 20 is {top_similarity / random_similarity:.1f}x more self-similar")

So the shortlist is largely one family of molecules found over and over,
not twenty independent leads. These are diterpene esters, relatives of phorbol, and
several of them do have published anti-HIV activity, which is why the model likes
them.

> **Exercise:** Build a more useful shortlist by keeping only the best-scoring
> molecule from each Murcko scaffold (`modelling.murcko_scaffolds`). How many
> different scaffolds are there in the top 100?

## 8. A second opinion from the regression model

The other model from `purple_baseline_models` predicts pActivity directly instead of
a label. If two models trained differently agree on which molecules matter, that is
mildly reassuring. Where they disagree is more interesting.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regression = pd.read_csv("data/hiv1_regression.csv")
regression = regression[regression["inchikey"].isin(set(curated["inchikey"]))]
rows = curated["inchikey"].isin(set(regression["inchikey"])).to_numpy()
target = regression.set_index("inchikey").loc[curated.loc[rows, "inchikey"], "pactivity"]

regressor = RandomForestRegressor(
    n_estimators=100, max_features="sqrt", n_jobs=-1, random_state=RANDOM_SEED
).fit(X_train[rows], target.to_numpy())
screen["pactivity_pred"] = regressor.predict(X_screen)
print(f"trained on {int(rows.sum()):,} molecules with a measured pActivity")

Do the two models rank the natural products the same way?

In [ ]:
from scipy.stats import spearmanr

anpdb_screen = screen[screen["library"] == "ANPDB"]
agreement = spearmanr(anpdb_screen["p_active"], anpdb_screen["pactivity_pred"]).statistic
top_regressor = set(anpdb_screen.nlargest(20, "pactivity_pred")["source_id"])
print(f"rank agreement across ANPDB: {agreement:.3f}")
print(f"top 20 shared: {len(set(top['source_id']) & top_regressor)} of 20")

Closely, then. That is worth remembering: these are not two independent
opinions. They were trained on the same molecules with the same fingerprints, so
mostly they agree by construction.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.scatter(anpdb_screen["p_active"], anpdb_screen["pactivity_pred"], s=3, alpha=0.2,
           color=nc.purple, linewidths=0)
ax.axhline(THRESHOLD, color=nc.pink, linestyle="--")
stylia.label(ax, xlabel="P(active) from the classifier",
             ylabel="pActivity from the regressor",
             title=f"The two models agree (Spearman {agreement:.2f})")

There is one thing the regressor cannot do, and the plot shows it. Nothing
is predicted below about 4.4, even for molecules the classifier is sure about.

In [ ]:
print(f"regressor's training data ran from {target.min():.2f} to {target.max():.2f}")
print(f"its ANPDB predictions run from {anpdb_screen['pactivity_pred'].min():.2f} "
      f"to {anpdb_screen['pactivity_pred'].max():.2f}")
print(f"median predicted pActivity for a natural product: "
      f"{anpdb_screen['pactivity_pred'].median():.2f}")

A median prediction around 5.4 means "the typical African natural product
is a 4 micromolar HIV inhibitor", which is plainly false. The reason is in the
training data: the regression set only contains molecules with a *measured* potency.
Compounds that did nothing were recorded as "no activity up to 50 uM" and never made
it in, so the regressor has never seen an inactive molecule and has no way to predict
one.

> **Note:** This is why the classifier is the better tool for screening. It was trained
> with inactive molecules, so it can say no. The regressor is useful for ordering
> molecules, not for judging them.

Finally, the molecules the two models disagree about most.

In [ ]:
ranked = anpdb_screen.assign(
    rank_classifier=(-anpdb_screen["p_active"]).rank(method="min"),
    rank_regressor=(-anpdb_screen["pactivity_pred"]).rank(method="min"),
)
ranked["gap"] = ranked["rank_regressor"] - ranked["rank_classifier"]
columns = ["mol_name", "rank_classifier", "rank_regressor", "max_similarity"]
ranked.nlargest(5, "gap")[columns].round(2)

These are molecules the classifier likes and the regressor does not. Many
of them are common plant triterpene acids, the sort of compound that turns up as a
weak hit in almost every assay ever run. The regressor pushing them down is a point
in its favour, and a reminder to look at disagreements rather than average them away.

## Summary

- Cleaned ANPDB and DrugBank the same way the training data was cleaned, leaving
  about 10,500 natural products and 11,200 drugs to score.
- Found that most natural products sit far from anything the model was trained on,
  and removed the 191 it had already seen. Without that step the five highest-scoring
  natural products would all have been molecules from the training data.
- Scored both libraries. Natural products score lower than drugs overall, and the
  top of the ANPDB list is one family of diterpene esters rather than twenty separate
  leads.
- Saw why a check on the approved HIV drugs proves nothing: all 26 are in the
  training set.
- Compared the classifier with the regressor. They agree closely, so they are one
  opinion rather than two, and the regressor cannot predict a molecule to be inactive.

**Next:** the hits here need a cytotoxicity counter-screen before any of them could be
called antiviral, since the training assay rewards anything that keeps cells alive.
`purple_hdac1_feasibility` applies the same pipeline to a second target.